In [3]:
import pandas as pd
import numpy as np

# Seed for reproducibility
np.random.seed(42)
num_rows = 10000

# 1. Generate realistic but messy raw data
data = {
    "transaction_id": [f"TXN_{100000 + i}" for i in range(num_rows)],
    "user_id": np.random.randint(100, 1050, size=num_rows),
    "plan_tier": np.random.choice(["Free", "Premium", "Enterprise", "Unknown"], size=num_rows, p=[0.5, 0.3, 0.15, 0.05]),
    "monthly_spend": np.random.choice([0.00, 29.99, 499.00, np.nan], size=num_rows, p=[0.45, 0.30, 0.20, 0.05]),
    "api_calls": np.random.negative_binomial(n=10, p=0.001, size=num_rows), # Simulates highly skewed web traffic
    "response_latency_ms": np.random.normal(loc=120, scale=45, size=num_rows), # System performance metrics
    "error_flag": np.random.choice([0, 1], size=num_rows, p=[0.96, 0.04]) # System failures
}

# 2. Inject dirty anomalies deliberately for you to clean!
df_raw = pd.DataFrame(data)

# Inject negative latency errors (impossible in real life, representing system glitches)
bad_latency_indices = np.random.choice(num_rows, size=150, replace=False)
df_raw.loc[bad_latency_indices, "response_latency_ms"] = -999.0

# Inject absurdly high API call outliers (simulating DDoS or loop bugs)
spam_indices = np.random.choice(num_rows, size=30, replace=False)
df_raw.loc[spam_indices, "api_calls"] = 999999

# Save to CSV
df_raw.to_csv("raw_saas_traffic.csv", index=False)
print("✅ Successfully generated 'raw_saas_traffic.csv' with 10,000 messy rows!")

✅ Successfully generated 'raw_saas_traffic.csv' with 10,000 messy rows!


In [4]:
mycsv = "raw_saas_traffic.csv"
df = pd.read_csv(mycsv)

In [5]:
print(df.shape)

(10000, 7)


In [6]:
print("null vlaues")
df.isnull()

null vlaues


,transaction_id,user_id,plan_tier,monthly_spend,api_calls,response_latency_ms,error_flag
0,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...
9995,False,False,False,False,False,False,False
9996,False,False,False,False,False,False,False
9997,False,False,False,False,False,False,False
9998,False,False,False,False,False,False,False


In [7]:
sum_null = df.isnull().sum()
sum_null

transaction_id           0
user_id                  0
plan_tier                0
monthly_spend          504
api_calls                0
response_latency_ms      0
error_flag               0
dtype: int64

In [8]:
df["monthly_spend"] = df["monthly_spend"].fillna(0.0)
print(df["monthly_spend"])
print("============================= null value figures=========================")
monthly_spend_null = df.isnull().sum()
monthly_spend_null

0         0.00
1         0.00
2        29.99
3        29.99
4         0.00
         ...  
9995    499.00
9996      0.00
9997      0.00
9998      0.00
9999      0.00
Name: monthly_spend, Length: 10000, dtype: float64
============================= null value figures=========================


transaction_id         0
user_id                0
plan_tier              0
monthly_spend          0
api_calls              0
response_latency_ms    0
error_flag             0
dtype: int64

In [20]:
valid_latencies =  df[df['response_latency_ms'] >=0 ]["response_latency_ms"]
median_val = valid_latencies.median()
df["response_latency_ms"] = np.where(
    df["response_latency_ms"] < 0, median_val, df["response_latency_ms"]
)


In [14]:
security_alerts = df[(df['api_calls'] > 500000) | (df['error_flag'] == 1 )]
print("security alerts", len(security_alerts))

security alerts 430


In [18]:
plan_total = df.groupby('plan_tier')['api_calls'].sum()
print(plan_total)



plan_tier
Enterprise    19944258
Free          67421346
Premium       37232947
Unknown        5013507
Name: api_calls, dtype: int64


In [19]:
plan_average = df.groupby('plan_tier')['response_latency_ms'].mean()
print(plan_average)


plan_tier
Enterprise    122.614317
Free          118.963831
Premium       120.800685
Unknown       116.091745
Name: response_latency_ms, dtype: float64
